# Metadati Import via API

This notebook replicates the logic in `mmt_motm/importers/db_import.py`
but uses HTTP POST calls to the motm REST API instead of direct DB access.

**API base URL**: `http://localhost:8000/motm/api/`

Before running, make sure the Django server is running and you have
a valid session or credentials for authenticated POST requests.

In [ ]:
import json
import os
from pathlib import Path

import requests

In [ ]:
# Configuration — adjust as needed
BASE_URL = os.environ.get("MMT_API_URL", "http://localhost:8000/motm/api")
SESSION = requests.Session()
# If using Django session auth with CSRF, you can login first:
# SESSION.get("http://localhost:8000/admin/login/")
# SESSION.post("http://localhost:8000/admin/login/", data={"username": "...", "password": "...", "csrfmiddlewaretoken": SESSION.cookies["csrftoken"]})

PARSED_DIR = Path.cwd().parent / "data" / "parsed"

## Utility functions

In [ ]:
RELATIONSHIP_MAP = {
    "vater": "father",
    "mutter": "mother",
    "schwester": "sister",
    "bruder": "brother",
    "großvater": "grandfather",
    "grossvater": "grandfather",
    "großvater väterlicherseits": "paternal grandfather",
    "grossvater väterlicherseits": "paternal grandfather",
    "großvater väterl": "paternal grandfather",
    "großmutter väterlicherseits": "paternal grandmother",
    "großmutter väterl": "paternal grandmother",
    "grossmutter väterlicherseits": "paternal grandmother",
    "großmutter mütterlicherseits": "maternal grandmother",
    "grossmutter mütterlicherseits": "maternal grandmother",
    "großmutter mütt": "maternal grandmother",
    "großvater mütterlicherseits": "maternal grandfather",
    "großvater mütt": "maternal grandfather",
    "tante": "aunt",
    "onkel": "uncle",
    "onkel väterl": "paternal uncle",
    "bruder aus erster ehe": "brother from the first marriage",
    "halbschwester": "half sister",
    "1 frau erste ehe": "first wife",
    "stiefmutter": "stepmother",
    "cousin": "cousin",
    "vetter": "cousin",
    "cousine": "cousin",
    "base": "cousin"
}


def is_empty(value):
    return value in [None, "", "-", "–", "—"]


def clean_label(label):
    label = label.strip().lower()
    label = label.replace("(", "").replace(")", "")
    label = label.replace(".", "")
    label = " ".join(label.split())
    return label


def parse_name(full_name):
    if not full_name:
        return None, None
    full_name = full_name.strip()
    if "," in full_name:
        parts = full_name.split(",")
        family_name = parts[0].strip()
        given_name = parts[1].strip() if len(parts) > 1 else "UNKNOWN"
        return given_name, family_name
    parts = full_name.split()
    if len(parts) > 1:
        return parts[0], " ".join(parts[1:])
    return full_name, "UNKNOWN"


def api_get(endpoint, params=None):
    """GET from the API, return JSON list (handles pagination)."""
    resp = SESSION.get(f"{BASE_URL}/{endpoint}/", params=params)
    resp.raise_for_status()
    data = resp.json()
    if isinstance(data, dict) and "results" in data:
        return data["results"]
    return data


def api_post(endpoint, payload):
    """POST to the API, return created object as dict."""
    resp = SESSION.post(f"{BASE_URL}/{endpoint}/", json=payload)
    resp.raise_for_status()
    return resp.json()

## Region hierarchy

In [ ]:
def get_or_create_region_hierarchy(region_list):
    """Create/get nested regions via API, return innermost region id."""
    if not region_list:
        return None
    parent_id = None
    for name in region_list:
        if is_empty(name):
            continue
        existing = api_get("regions", params={"search": name})
        match = next((r for r in existing if r["name"] == name), None)
        if match:
            region_id = match["id"]
            if parent_id and match.get("part_of") != parent_id:
                SESSION.patch(
                    f"{BASE_URL}/regions/{region_id}/",
                    json={"part_of": parent_id},
                )
        else:
            created = api_post("regions", {"name": name, "part_of": parent_id})
            region_id = created["id"]
        parent_id = region_id
    return parent_id

## Location (birth place)

In [ ]:
def get_or_create_location(bp):
    """Create or retrieve a LocationPoint via the API."""
    if is_empty(bp.get("name")):
        return None

    existing = api_get("locations", params={"search": bp["name"]})
    match = next((loc for loc in existing if loc["current_name"] == bp["name"]), None)

    if match:
        loc_id = match["id"]
        if not match.get("region"):
            region_id = get_or_create_region_hierarchy(bp.get("regions"))
            if region_id:
                SESSION.patch(
                    f"{BASE_URL}/locations/{loc_id}/",
                    json={"region": region_id},
                )
        return loc_id

    payload = {"current_name": bp["name"]}
    if bp.get("wikidata_id"):
        payload["wikidata_id"] = bp["wikidata_id"]
    if bp.get("coordinates"):
        payload["latitude"] = bp["coordinates"]["lat"]
        payload["longitude"] = bp["coordinates"]["lon"]

    region_id = get_or_create_region_hierarchy(bp.get("regions"))
    if region_id:
        payload["region"] = region_id

    created = api_post("locations", payload)
    return created["id"]

## Person matching

In [ ]:
def get_or_create_person(data, identifier=None):
    """Create or match a Person via the API. Returns person id or None."""
    given_name = data.get("given_name")
    family_name = data.get("family_name")

    if is_empty(given_name) and is_empty(family_name):
        return None

    # Match on identifier
    if identifier:
        existing = api_get("persons", params={"search": identifier})
        match = next((p for p in existing if p.get("identifier") == identifier), None)
        if match:
            return match["id"]
        payload = {
            "identifier": identifier,
            "given_name": given_name or "",
            "family_name": family_name or "",
        }
        if data.get("birth_date"):
            payload["birth_date"] = data["birth_date"]
        if data.get("gender"):
            payload["gender"] = data["gender"]
        if data.get("attributes", {}).get("ns_persecution_group"):
            payload["description"] = data["attributes"]["ns_persecution_group"]
        created = api_post("persons", payload)
        return created["id"]

    # Match on name + birth_date
    if not is_empty(given_name) and not is_empty(family_name):
        existing = api_get("persons", params={"search": family_name})
        for p in existing:
            if p["given_name"] == given_name and p["family_name"] == family_name:
                if data.get("birth_date") and p.get("birth_date") == data["birth_date"]:
                    return p["id"]
                if not data.get("birth_date"):
                    return p["id"]

    payload = {
        "given_name": given_name or "",
        "family_name": family_name or "",
    }
    if data.get("birth_date"):
        payload["birth_date"] = data["birth_date"]
    created = api_post("persons", payload)
    return created["id"]

## Interview

In [ ]:
def create_interview(person_id, data):
    """Create an Interview via the API."""
    if is_empty(data.get("archive_id")):
        return None

    archive_id = data["archive_id"]

    existing = api_get("interviews", params={"search": archive_id})
    match = next((i for i in existing if i["archive_id"] == archive_id), None)
    if match:
        return match["id"]

    interviewer_id = None
    if not is_empty(data.get("interviewer")):
        given, family = parse_name(data["interviewer"])
        interviewer_id = get_or_create_person({
            "given_name": given,
            "family_name": family,
        })

    payload = {
        "archive_id": archive_id,
        "interviewee": person_id,
        "interview_type": data.get("type") or "",
        "date": data.get("date"),
        "place": data.get("place") or "",
    }
    if interviewer_id:
        payload["interviewer"] = interviewer_id

    created = api_post("interviews", payload)
    return created["id"]

## Relationship

In [ ]:
def get_or_create_relationship_type(label):
    """Get or create a RelationshipType via the API."""
    if is_empty(label):
        return None

    label_clean = clean_label(label)
    mapped = RELATIONSHIP_MAP.get(label_clean, label_clean)

    existing = api_get("relationship-types", params={"search": mapped})
    match = next((rt for rt in existing if rt["name"] == mapped), None)
    if match:
        return match["id"]

    created = api_post("relationship-types", {
        "name": mapped,
        "original_label": label.strip(),
    })
    return created["id"]


def create_relationship(main_person_id, data, fallback_family_name):
    """Create a Relationship via the API."""
    if is_empty(data.get("relation")):
        return None

    given_name = data.get("given_name")
    family_name = data.get("family_name")
    if is_empty(family_name):
        family_name = fallback_family_name

    related_id = get_or_create_person({
        "given_name": given_name,
        "family_name": family_name,
        "birth_date": data.get("birth_date"),
    })
    if not related_id:
        return None

    rel_type_id = get_or_create_relationship_type(data.get("relation"))
    if not rel_type_id:
        return None

    # Check for existing relationship
    existing = api_get("relationships")
    for r in existing:
        if (r["person_from"] == main_person_id
                and r["person_to"] == related_id
                and r["relationship_type"] == rel_type_id):
            return r["id"]

    created = api_post("relationships", {
        "relationship_type": rel_type_id,
        "person_from": main_person_id,
        "person_to": related_id,
        "description": data.get("notes") or "",
    })
    return created["id"]

## Import a single record

In [ ]:
def import_record(record):
    """Import a full person record (parsed JSON) via API calls."""
    person_id = get_or_create_person(
        record["person"],
        identifier=record.get("identifier"),
    )
    if not person_id:
        print("Skipped: invalid person")
        return None

    # Birth place
    location_id = get_or_create_location(record.get("birth_place", {}))
    if location_id:
        SESSION.patch(
            f"{BASE_URL}/persons/{person_id}/",
            json={"birth_place": location_id},
        )

    # Interviews
    for interview_data in record.get("interviews", []):
        create_interview(person_id, interview_data)

    # Family / relationships
    person_data = SESSION.get(f"{BASE_URL}/persons/{person_id}/").json()
    main_family_name = person_data.get("family_name", "")
    for family_data in record.get("family", []):
        create_relationship(person_id, family_data, main_family_name)

    return person_id

## Import from file

In [ ]:
def import_from_file(path):
    with open(path, encoding="utf-8") as f:
        record = json.load(f)
    person_id = import_record(record)
    if person_id:
        print(f"\u2705 Imported: {path}")
    else:
        print(f"\u26a0\ufe0f Skipped: {path}")
    return person_id

## Import all parsed JSON files

In [ ]:
def import_all(directory=None):
    if directory is None:
        directory = PARSED_DIR
    directory = Path(directory)
    results = []
    for filepath in sorted(directory.glob("*.json")):
        try:
            person_id = import_from_file(filepath)
            if person_id:
                results.append(person_id)
        except Exception as e:
            print(f"\u274c ERROR in {filepath.name}: {e}")
    print(f"\n\u2705 Imported {len(results)} valid records")
    return results

## Run the import

Uncomment the appropriate line below to import a single file or all files.

In [ ]:
# Single file:
# import_from_file(PARSED_DIR / "IS_S_00142.person.json")

# All files:
# import_all()